# **🏋🏽 Wearable Biomechanics - Dataset Export & Reports**

### 🔧 Colab Setup & Library Init

In [12]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure renderers
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "cleannjerk_dataset" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.export import clean_export_dir, export_dataset, load_and_deduplicate_entries, build_id_mapping_df

source_base_dir = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}
output_dir = "/content/drive/MyDrive/Academic/Research/Wearable Sensing/Wearable Biomechanics/Clean&JerkDataset" # @param {type:"string"}
mapping_csv_path = os.path.join(repo_root, "docs", "export_id_mapping.csv")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 🔍 Preview Chronological ID Mapping

In [13]:
# @title 🔍 Preview Chronological ID Mapping
records, sub_map, _ = load_and_deduplicate_entries(
    os.path.join(source_base_dir, "index"),
    existing_mapping_csv=mapping_csv_path if os.path.exists(mapping_csv_path) else None,
)
mdf = build_id_mapping_df(records)
print(f"Total unique Sets: {len(mdf)} | Total Participants: {len(sub_map)}")
display_table(mdf, clean_columns=False)

Total unique Sets: 54 | Total Participants: 19


      set_id participant_id            source_tier  \
0   set_0001          P0001        complete_trials   
1   set_0002          P0001        complete_trials   
2   set_0003          P0002        complete_trials   
3   set_0004          P0003        complete_trials   
4   set_0005          P0003        complete_trials   
5   set_0006          P0004        complete_trials   
6   set_0007          P0004        complete_trials   
7   set_0008          P0005        complete_trials   
8   set_0009          P0005        complete_trials   
9   set_0010          P0006        complete_trials   
10  set_0011          P0006        complete_trials   
11  set_0012          P0007  fault_tolerant_trials   
12  set_0013          P0007  fault_tolerant_trials   
13  set_0014          P0008        complete_trials   
14  set_0015          P0008        complete_trials   
15  set_0016          P0008        complete_trials   
16  set_0017          P0009        complete_trials   
17  set_0018          P0009 

### 🧹 Clean Export Folder (Optional Reset)

In [ ]:
# @title 🧹 Clean Export Folder { display-mode: "form" }
confirm_clean = False # @param {type:"boolean"}
if confirm_clean:
    clean_export_dir(output_dir)
else:
    print("ℹ️ Set confirm_clean = True to wipe previously exported files in output_dir.")

### 🚀 Run Dataset Export (Batch or Incremental)

In [14]:
# @title 🚀 Run Dataset Export (Dry Run or Real) { display-mode: "form" }
dry_run = True # @param {type:"boolean"}
only_new = False # @param {type:"boolean"}
copy_videos = False # @param {type:"boolean"}

exported = export_dataset(
    source_base_dir=source_base_dir,
    output_dir=output_dir,
    mapping_csv_path=mapping_csv_path,
    copy_videos=copy_videos,
    only_new=only_new,
    dry_run=dry_run,
)

display_table(exported["diagnostics"], clean_columns=False)
if not dry_run:
    print(f"\n✅ Export complete to: {output_dir}")
    print(f"📝 ID mapping saved to: {mapping_csv_path}")

[DRY RUN] set_0001 (complete_trials_0023): sensors_ok=23/26, markers=48 (ann=3), videos=(F:False, S:True), has_faults=False | faulty=RAD (missing); RAP (missing); RKD (missing)
[DRY RUN] set_0002 (complete_trials_0024): sensors_ok=22/26, markers=60 (ann=3), videos=(F:False, S:True), has_faults=False | faulty=LED (missing); RAD (missing); RAP (missing); RKD (missing)
[DRY RUN] set_0003 (complete_trials_0025): sensors_ok=26/26, markers=72 (ann=3), videos=(F:False, S:True), has_faults=False
[DRY RUN] set_0004 (complete_trials_0018): sensors_ok=26/26, markers=60 (ann=3), videos=(F:False, S:True), has_faults=False
[DRY RUN] set_0005 (complete_trials_0019): sensors_ok=26/26, markers=60 (ann=3), videos=(F:False, S:True), has_faults=False
[DRY RUN] set_0006 (complete_trials_0020): sensors_ok=25/26, markers=60 (ann=3), videos=(F:False, S:True), has_faults=False | faulty=RWP (freeze)
[DRY RUN] set_0007 (complete_trials_0022): sensors_ok=25/26, markers=60 (ann=3), videos=(F:False, S:True), has_fa

      set_id participant_id             source_entry_id  \
0   set_0001          P0001        complete_trials_0023   
1   set_0002          P0001        complete_trials_0024   
2   set_0003          P0002        complete_trials_0025   
3   set_0004          P0003        complete_trials_0018   
4   set_0005          P0003        complete_trials_0019   
5   set_0006          P0004        complete_trials_0020   
6   set_0007          P0004        complete_trials_0022   
7   set_0008          P0005        complete_trials_0026   
8   set_0009          P0005        complete_trials_0027   
9   set_0010          P0006        complete_trials_0028   
10  set_0011          P0006        complete_trials_0029   
11  set_0012          P0007  fault_tolerant_trials_0006   
12  set_0013          P0007  fault_tolerant_trials_0005   
13  set_0014          P0008        complete_trials_0009   
14  set_0015          P0008        complete_trials_0007   
15  set_0016          P0008        complete_trials_0008 

### ✅ Inspect Exported Summary Tables

In [15]:
# @title ✅ Inspect Exported Summary Tables
display_table(exported["participants"], clean_columns=False)
display_table(exported["sets"], clean_columns=False)
display_table(exported["devices"], clean_columns=False)
display_table(exported["phase_markers"].head(50), clean_columns=False)

   participant_id                       name age_years sex height_m  \
0           P0001                      Enuka      <NA>  NA     <NA>   
1           P0002      Thevindu Samarasekera        25  NA     1.69   
2           P0003             Achalani Royal        23  NA      1.6   
3           P0004                    Pandula      <NA>  NA     <NA>   
4           P0005          Induwara Liyanage        21  NA      1.7   
5           P0006                   Duleetha      <NA>  NA     <NA>   
6           P0007         Hivindu Kirulapana        22  NA     1.74   
7           P0008          Yewin Dissanayake        22  NA      1.7   
8           P0009         Yasith Dissanayake        22  NA     1.78   
9           P0010  J W K Mahasen Jayasundara        24  NA     1.78   
10          P0011      Nirmal Samarawickrama        22  NA     1.73   
11          P0012   J. M. Sudeshe Jayasinghe        23  NA     1.56   
12          P0013          Dulan Jayasooriya        21  NA      1.8   
13    

      set_id participant_id session_date  load_kg  lifts_total  lifts_usable  \
0   set_0001          P0001   2025-09-16     20.0            5             5   
1   set_0002          P0001   2025-09-16     20.0            5             5   
2   set_0003          P0002   2025-09-17     20.0            5             5   
3   set_0004          P0003   2025-10-01     20.0            5             5   
4   set_0005          P0003   2025-10-01     20.0            5             5   
5   set_0006          P0004   2025-10-01     20.0            5             5   
6   set_0007          P0004   2025-10-01     20.0            5             5   
7   set_0008          P0005   2025-10-30     20.0            5             5   
8   set_0009          P0005   2025-10-30     20.0            5             5   
9   set_0010          P0006   2025-10-30     20.0            5             5   
10  set_0011          P0006   2025-10-30     20.0            5             5   
11  set_0012          P0007   2026-05-08

   device_id  imu_index short_code          anatomical_name    side     joint  \
0         01          1        RKP      right_knee_proximal   right      knee   
1         01          2        RHD         right_hip_distal   right       hip   
2         02          1        REP     right_elbow_proximal   right     elbow   
3         02          2        RSD    right_shoulder_distal   right  shoulder   
4         03          1        LKP       left_knee_proximal    left      knee   
5         03          2        LHD          left_hip_distal    left       hip   
6         04          1        LEP      left_elbow_proximal    left     elbow   
7         04          2        LSD     left_shoulder_distal    left  shoulder   
8      05/11          1        LHP        left_hip_proximal    left       hip   
9      05/11          2        RHP       right_hip_proximal   right       hip   
10     05/11          3         LS              lower_spine  center     spine   
11        06          1     

      set_id  lift_id        marker_name  n_annotators  mean_frame  sd_frames  \
0   set_0001        1              start             3      0.0000     0.0000   
1   set_0001        1              setup             3     36.3333     4.0415   
2   set_0001        1         first_pull             3    137.3333     0.5774   
3   set_0001        1        second_pull             3    151.0000     3.0000   
4   set_0001        1              clean             3    157.3333     2.5166   
5   set_0001        1  intermediate_rest             3    230.3333     6.1101   
6   set_0001        1         jerk_split             3    257.0000     1.0000   
7   set_0001        1      jerk_recovery             3    337.6667     1.5275   
8   set_0001        1      overhead_hold             3    361.3333     2.5166   
9   set_0001        1            descend             3    362.0000     2.6458   
10  set_0001        1          post_rest             3    448.3333     5.5076   
11  set_0001        1       